분산,표준편차 구하기

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
  .master("local[*]")\
  .getOrCreate()
sc = spark.sparkContext

In [2]:
data = sc.parallelize([1,2,3,4,5])
stats = data.map(lambda x: (x, x**2, 1)).reduce(lambda a,b : (a[0]+b[0], a[1]+b[1], a[2]+b[2]))

sum_x, sum_xx, n = stats

In [3]:
import numpy as np
mean = sum_x/n
variance = (sum_xx/n) - (mean**2)
std_dev = np.sqrt(variance)
print(f"Mean: {mean}, Variance: {variance}, Std: {std_dev}")

Mean: 3.0, Variance: 2.0, Std: 1.4142135623730951


KNN-단순거리계산
(2,2)와 가장 가까운 점 3개 찾는 문제

In [4]:
# 전체 데이터
points = sc.parallelize([(1, 1), (5, 5), (1, 2), (10, 10), (2, 1)])
query = np.array([2, 2])
b_query = sc.broadcast(query)

K = 3

In [5]:
# 1. 거리 계산(MAP)
# 결과 (거리, 좌표)
distances = points.map(lambda p:(
    np.linalg.norm(np.array(p)-b_query.value), p
))
# 상위 k개 추출
k_nearest = distances.takeOrdered(K, key=lambda x: x[0])

print(f"Top {K} nearest points: {k_nearest}")

Top 3 nearest points: [(np.float64(1.0), (1, 2)), (np.float64(1.0), (2, 1)), (np.float64(1.4142135623730951), (1, 1))]
